In [13]:
import gymnasium as gym
from gymnasium import Env
from gymnasium.spaces import Discrete, Box, Dict, Tuple, MultiBinary, MultiDiscrete

import numpy as np
import random
import os

from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.evaluation import evaluate_policy

In [ ]:
Discrete(3)

In [3]:
Box(0,1, shape =(3,3))

Box(0.0, 1.0, (3, 3), float32)

In [4]:
Tuple((Discrete(3), Box(0,1, shape =(3,3))))

Tuple(Discrete(3), Box(0.0, 1.0, (3, 3), float32))

In [ ]:
Dict({'heaight':Discrete(2), "speed":Box(0,100,shape=(1,))})

{'heaight': np.int64(0), 'speed': array([67.37801], dtype=float32)}

In [10]:
MultiBinary(4).sample()

array([1, 0, 1, 0], dtype=int8)

In [9]:
MultiDiscrete([5,2,2]).sample()

array([1, 1, 1])

Build an Environement
-Build an agent to give us the best shower possible
-Random temperature fluxuation
-Best temps between 37 and 39

In [38]:
class ShowerEnv(Env):
    def __init__(self):
        self.action_space = Discrete(3)
        self.observation_space = Box(low=np.array([0]), high = np.array([100]))
        self.state = 38 + random.randint(-3,3)
        self.shower_length = 60
        pass
    def step(self, action):
        self.state += action-1
        self.shower_length -=1
        if self.state >=37 and self.state <=39:
            reward = 1
        else:
            reward = -1
            
        if self.shower_length <=0:
            done = True
        else:
            done = False
            
        truncated = self.shower_length <= 0
        info = {}
        
        return self.state, reward, done, truncated, info
            
    def render(self):
        #implement viz
        pass
    def reset(self, seed = None, options = None):
        super().reset(seed=seed)
        self.state = np.array([38 + random.randint(-3, 3)], dtype=float)
        self.shower_length = 60
        return self.state, {}


In [39]:
env = ShowerEnv()

In [35]:
episodes = 5
for episode in range(1, episodes+1):
    obs = env.reset()
    done = False
    score = 0
    while not done:
        env.render()
        action = env. action_space.sample()
        obs, reward, done, trunc, info = env.step(action)
        score+=reward
    print('Episodes:{} Score:{}'.format(episode, score))
env.close()

Episodes:1 Score:8
Episodes:2 Score:-22
Episodes:3 Score:-22
Episodes:4 Score:6
Episodes:5 Score:-60


In [40]:
log_path = os.path.join('Training', 'Logs')
model = PPO('MlpPolicy', env, verbose=1, tensorboard_log=log_path)

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


In [42]:
model.learn(total_timesteps=100000)

Logging to Training/Logs/PPO_13
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 60       |
|    ep_rew_mean     | -33.8    |
| time/              |          |
|    fps             | 8662     |
|    iterations      | 1        |
|    time_elapsed    | 0        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 60          |
|    ep_rew_mean          | -30         |
| time/                   |             |
|    fps                  | 5661        |
|    iterations           | 2           |
|    time_elapsed         | 0           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.009856131 |
|    clip_fraction        | 0.0267      |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.06       |
|    explained_variance   | -1.87e-05   

In [43]:
shower_path = os.path.join('Training', 'Saved Models', 'PPO_Shower_Model')
model.save(shower_path)

In [44]:
del model

In [47]:
model = PPO.load(shower_path, env)

Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


In [48]:
evaluate_policy(model, env, n_eval_episodes=10, render = True)

(np.float64(59.4), np.float64(0.9165151389911679))